# Gate A — SmolLM2 transfer (Kaggle T4×2)

Week 1. **No cache.** Never P100. Never bf16.

1. New notebook → accelerator **GPU T4 × 2**. Internet **on**.
2. Run setup, then convert → copy-only PPL → Taylor-Calibrate PPL → transfer.
3. Checkpoints under `/kaggle/working/checkpoints/gate-a` so they survive a kernel interrupt.
4. Taylor-Calibrate itself is not checkpointed; re-run that cell if the session dies. Transfer uses `--resume auto`.
5. After MSE falls and calibrated PPL ≪ copy-only, fill `docs/decisions/gate-A.md`.

In [ ]:
import os
from pathlib import Path

REPO = "https://github.com/Caedral-ai/notrehybrid.git"
cwd = Path.cwd()

if (cwd / "setup_kaggle.sh").exists():
    root = cwd
elif (cwd / "notrehybrid" / "setup_kaggle.sh").exists():
    root = cwd / "notrehybrid"
else:
    !git clone --depth 1 {REPO} notrehybrid
    root = cwd / "notrehybrid"

os.chdir(root)
print("repo root:", root)
!bash setup_kaggle.sh

In [ ]:
!python -m pytest tests/test_surgery.py tests/test_hybrid_block.py -q

## Convert + init

HF SmolLM2 → FLA teacher, then copy-only student vs Taylor-Calibrate. Paths live on `/kaggle/working` so they survive an interrupt.

In [ ]:
WORK = "/kaggle/working"
TEACHER = f"{WORK}/teachers/SmolLM2-360M"
COPY = f"{WORK}/checkpoints/gate-a/init-copy"
TAYLOR = f"{WORK}/checkpoints/gate-a/init-taylor"
CKPT = f"{WORK}/checkpoints/gate-a"
CFG = "configs/smollm2_360m/gate_a.yaml"
print(TEACHER, COPY, TAYLOR)

In [ ]:
!python -m notre.convert.convert_smollm2 --hf HuggingFaceTB/SmolLM2-360M --out {TEACHER}

In [ ]:
!python -m notre.convert.init_student --cfg {CFG} --output {COPY} --teacher {TEACHER}

In [ ]:
!python -m notre.eval.ppl --ckpt {COPY} --tokenizer {TEACHER}

## Taylor-Calibrate

fp16 wrap of `apply_taylor_calibrate`. Not resumed — re-run this cell if the kernel dies.

In [ ]:
!python -m notre.convert.taylor_calibrate --cfg {CFG} --output {TAYLOR} --teacher {TEACHER} --hf-teacher HuggingFaceTB/SmolLM2-360M

In [ ]:
!python -m notre.eval.ppl --ckpt {TAYLOR} --tokenizer {TEACHER}

## Transfer MSE probe (quota)

Dry run (3 steps), then **20 minutes** only. The full ~5M tokens waits until this probe shows finite, falling MSE. `--resume auto` continues `mse.csv`. No collision cache.

In [ ]:
!python -m notre.convert.transfer --cfg {CFG} --teacher {TEACHER} --student-init {TAYLOR} --ckpt-dir {CKPT} --max-steps 3 --save-every 1 --keep-last 2

In [ ]:
# 20-minute probe. Full 5M is a later run once MSE is finite and falling.
!python -m notre.convert.transfer --cfg {CFG} --teacher {TEACHER} --student-init {TAYLOR} --ckpt-dir {CKPT} --minutes 20 --save-every 50 --keep-last 2 --resume auto

In [ ]:
from pathlib import Path
p = Path("/kaggle/working/checkpoints/gate-a/transfer/mse.csv")
print(p.read_text() if p.exists() else "mse.csv missing")
print("Fill docs/decisions/gate-A.md: copy-only PPL, Taylor PPL, initial vs final MSE.")